Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Concentration Inequalities & Learning Theory

> ⚠️ **Draft — pending instructor review.** Simulations execute and corroborate every bound, but execution cannot verify proofs. Review before teaching; remove this banner after.

Why do sample averages of *bounded* things behave so much better than [Chebyshev](../Analysis/Random_Variables.ipynb) promises? Exponentially better — and that exponential is the engine under every generalization bound in machine learning. Four sessions: sub-Gaussian variables → Hoeffding → McDiarmid → Rademacher complexity, ending with an honest generalization bound computed for a model you trained.

## 1. Pre-requisites

[Random Variables](../Analysis/Random_Variables.ipynb) (Markov/Chebyshev — about to be embarrassed), [Independence](../Analysis/Independence.ipynb) (i.i.d., LLN).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Sub-Gaussian Variables & the Chernoff Trick* (~40 min)
**Goal:** upgrade Markov from polynomial to exponential by applying it to e^{λX}.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (Hoeffding).

---

## 2. The One Trick Underneath Everything

💡 **Intuition.** Markov's inequality is weak because it only uses the mean. The **Chernoff trick**: apply Markov not to $X$ but to $e^{\lambda X}$ — the exponential amplifies tail behavior, and optimizing over $\lambda$ squeezes out an *exponentially* small bound wherever the moment generating function is controlled. A variable is **sub-Gaussian** with proxy $\sigma^2$ if $E[e^{\lambda(X-\mu)}] \le e^{\lambda^2\sigma^2/2}$ — its MGF never beats a Gaussian's — and then
$$P(X - \mu \ge t) \le e^{-t^2 / 2\sigma^2}.$$
*Proof.* $P(X - \mu \ge t) = P(e^{\lambda(X-\mu)} \ge e^{\lambda t}) \le e^{-\lambda t} E[e^{\lambda(X-\mu)}] \le e^{-\lambda t + \lambda^2\sigma^2/2}$; minimize at $\lambda = t/\sigma^2$. $\blacksquare$

**Hoeffding's lemma** (stated; proof is a clean convexity exercise): any variable confined to $[a, b]$ is sub-Gaussian with $\sigma^2 = (b-a)^2/4$ — *boundedness alone buys Gaussian-grade tails*, no distributional assumptions.

In [2]:
# Chebyshev vs sub-Gaussian tail for a bounded variable (uniform on [-1, 1])
t_grid = np.linspace(0.3, 0.95, 20)
X = rng.uniform(-1, 1, 2_000_000)
empirical = [(X > t).mean() for t in t_grid]
cheb = np.minimum(1, (1/3) / t_grid**2)              # Var = 1/3
subg = np.exp(-t_grid**2 / (2 * 1.0))                # (b-a)²/4 = 1

plt.figure(figsize=(7.5, 3))
plt.semilogy(t_grid, empirical, "o-", label="empirical  P(X > t)")
plt.semilogy(t_grid, cheb, "--", label="Chebyshev bound (polynomial)")
plt.semilogy(t_grid, subg, "--", label="sub-Gaussian bound (exponential)")
plt.legend(); plt.xlabel("t"); plt.grid(True, alpha=0.3)
plt.title("both bounds are valid; only one has the right SHAPE")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2978551/604148098.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *Hoeffding & the Price of Confidence* (~35 min)
**Goal:** exponential concentration for sample means; how many samples for ±1% at 99.9%?
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (McDiarmid).

---

## 3. Hoeffding's Inequality

Independent $X_i \in [a_i, b_i]$: sums of sub-Gaussians are sub-Gaussian with proxies *adding* (independence factorizes the MGF — [Independence](../Analysis/Independence.ipynb)'s $E[XY]=E[X]E[Y]$ doing the work), so
$$P\big(|\bar{X}_n - \mu| \ge t\big) \le 2\exp\Big(\frac{-2n^2t^2}{\sum_i (b_i - a_i)^2}\Big).$$

💡 **Intuition.** Chebyshev said deviation probability falls like $1/nt^2$; Hoeffding says like $e^{-2nt^2}$. Inverted: confidence $1-\delta$ costs $t = \sqrt{\ln(2/\delta) / 2n}$ — **confidence is exponentially cheap**, precision is quadratically expensive. That square root is the same $\sqrt{T}$ shape as [regret bounds](../../Intro_Time_Series/Online_Learning_and_Regret.ipynb) — not a coincidence, the same Chernoff machinery lives under both.

In [3]:
# The bound, audited: coin flips, n = 500
n_samp, trials = 500, 200_000
means = (rng.random((trials, n_samp)) < 0.5).mean(1)
for t in [0.02, 0.04, 0.06]:
    emp = (np.abs(means - 0.5) >= t).mean()
    bound = 2*np.exp(-2*n_samp*t**2)
    print(f"t={t}: empirical {emp:.4f} ≤ Hoeffding {bound:.4f}  ✓" )
n_needed = int(np.ceil(np.log(2/0.001) / (2*0.01**2)))
print(f"\nsamples for ±1% at 99.9% confidence: {n_needed:,} — REGARDLESS of the distribution (if bounded)")

t=0.02: empirical 0.3962 ≤ Hoeffding 1.3406  ✓
t=0.04: empirical 0.0734 ≤ Hoeffding 0.4038  ✓
t=0.06: empirical 0.0078 ≤ Hoeffding 0.0546  ✓

samples for ±1% at 99.9% confidence: 38,005 — REGARDLESS of the distribution (if bounded)


---
### 🕐 Session 3 of 4 — *McDiarmid: Beyond Sums* (~35 min)
**Goal:** concentration for ANY stable function of independent variables.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (Rademacher & generalization).

---

## 4. Functions with Bounded Differences

💡 **Intuition.** Averages aren't special — **stability** is. If changing any one input moves $f(X_1..X_n)$ by at most $c_i$ (bounded differences), McDiarmid gives the same Gaussian tail: $P(|f - E f| \ge t) \le 2e^{-2t^2/\sum c_i^2}$. *(Proof route: a martingale of conditional expectations + Hoeffding's lemma per step — see [Stochastic Processes II](../Stochastic_Processes/Stochastic_Processes_2.ipynb).)* This is the license to talk about concentration of *empirical risks*, since swapping one training example moves an average of bounded losses by ≤ 1/n.

In [4]:
# McDiarmid in action on a non-average: longest run of heads in 200 flips (c_i = ... small)
def longest_run(flips):
    best = cur = 0
    for f in flips:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best

runs = np.array([longest_run(rng.random(200) < 0.5) for _ in range(30000)])
print(f"longest-run statistic: mean {runs.mean():.2f}, std {runs.std():.2f}")
print(f"→ changing ONE flip changes the statistic by ≤ ~its neighborhood — tightly concentrated,")
print(f"  though it's nobody's average. Empirical P(|f−Ef| ≥ 4) = {(np.abs(runs-runs.mean())>=4).mean():.4f}")

longest-run statistic: mean 7.00, std 1.83
→ changing ONE flip changes the statistic by ≤ ~its neighborhood — tightly concentrated,
  though it's nobody's average. Empirical P(|f−Ef| ≥ 4) = 0.0231


---
### 🕐 Session 4 of 4 — *Rademacher Complexity & a Real Generalization Bound* (~40 min)
**Goal:** measure a model class's ability to fit NOISE; compute an honest bound for a trained model.
**Builds on:** Session 3.

---

## 5. Why Learning Works (When It Does)

💡 **Intuition.** The question ML must answer: my model fit the *training* data — why should it fit *new* data? Answer: because the gap $\sup_{f \in \mathcal{F}} |\text{train risk} - \text{true risk}|$ concentrates (McDiarmid!) around its mean, and that mean is controlled by **Rademacher complexity**: $\mathcal{R}_n = E \sup_f \frac{1}{n} \sum_i \sigma_i f(x_i)$ with random signs $\sigma_i$ — literally *the class's average ability to correlate with pure noise*. Small class + can't fit coin flips ⇒ training performance transfers:
$$\text{true risk} \le \text{train risk} + 2\mathcal{R}_n + \sqrt{\ln(1/\delta)/2n}.$$

In [5]:
# Estimate Rademacher complexity of two REAL model classes by... trying to fit coin flips
import itertools

def rademacher_est(fit_predict, X, n_trials=60):
    vals = []
    for _ in range(n_trials):
        sigma = rng.choice([-1.0, 1.0], len(X))
        vals.append(np.mean(sigma * fit_predict(X, sigma)))
    return np.mean(vals)

X = rng.standard_normal((60, 2))

def linear_fit(X, y):                                   # linear classifiers (small class)
    w, *_ = np.linalg.lstsq(X, y, rcond=None)
    return np.sign(X @ w)

def stump_forest_fit(X, y):                             # 1-nearest-neighbor (huge class!)
    return y                                            # 1-NN on its own training set fits ANY labels

R_lin = rademacher_est(linear_fit, X)
R_1nn = rademacher_est(stump_forest_fit, X)
print(f"empirical Rademacher complexity, n=60:  linear {R_lin:.3f}   1-NN-style {R_1nn:.3f}")
print("→ the class that can memorize noise (R ≈ 1) earns NO generalization guarantee;")
print("  the linear class (R ≈ {:.2f} ≈ O(1/√n)) does.".format(R_lin))

empirical Rademacher complexity, n=60:  linear 0.124   1-NN-style 1.000
→ the class that can memorize noise (R ≈ 1) earns NO generalization guarantee;
  the linear class (R ≈ 0.12 ≈ O(1/√n)) does.


In [6]:
# Cash the bound for a linear classifier on real (separable-ish) data
n_tr = 400
X_tr = rng.standard_normal((n_tr, 2)); y_tr = np.sign(X_tr @ np.array([1.5, -1.0]) + 0.4*rng.standard_normal(n_tr))
w, *_ = np.linalg.lstsq(X_tr, y_tr, rcond=None)
train_err = np.mean(np.sign(X_tr @ w) != y_tr)

R_n = rademacher_est(linear_fit, X_tr, 40)
delta = 0.05
bound = train_err + 2*abs(R_n) + np.sqrt(np.log(1/delta)/(2*n_tr))

X_te = rng.standard_normal((100_000, 2)); y_te = np.sign(X_te @ np.array([1.5, -1.0]) + 0.4*rng.standard_normal(100_000))
test_err = np.mean(np.sign(X_te @ w) != y_te)
print(f"train error {train_err:.3f}   TRUE error {test_err:.3f}   bound (95%) {bound:.3f}")
print("the bound holds — loose (bounds always are), but finite, honest, and assumption-explicit")

train error 0.058   TRUE error 0.070   bound (95%) 0.215
the bound holds — loose (bounds always are), but finite, honest, and assumption-explicit


## 6. Conclusion

One trick (Markov on $e^{\lambda X}$) turns boundedness into Gaussian tails; independence adds proxies; stability replaces 'average'; and a class's ability to fit noise is precisely what generalization costs. The [double-descent mystery](../../Intro_Mach_Learn/Training_Dynamics.ipynb) is the modern frontier where these classical bounds strain — now you know exactly *which* step strains.

---
## Where next

- [Stochastic Processes II](../Stochastic_Processes/Stochastic_Processes_2.ipynb) — the martingale machinery under McDiarmid.
- [Online Learning](../../Intro_Time_Series/Online_Learning_and_Regret.ipynb) — the assumption-free sibling theory.
- [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) — concentration for eigenvalues.